In [1]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    model_kwargs={"seed": 365}
)

In [4]:
TEMPLATE = """
System:
{description}

Human:
I've recently adopted a {pet}.
Could you suggest some {pet} names?
"""

prompt_template = PromptTemplate.from_template(template=TEMPLATE)

In [5]:
prompt_template

PromptTemplate(input_variables=['description', 'pet'], input_types={}, partial_variables={}, template="\nSystem:\n{description}\n\nHuman:\nI've recently adopted a {pet}.\nCould you suggest some {pet} names?\n")

In [6]:
prompt_value = prompt_template.invoke({
    'description': ''' The chatbot should reluctantly answer questions with sarcastic responses. ''',
    'pet': 'cat'
})

In [7]:
prompt_value

StringPromptValue(text="\nSystem:\n The chatbot should reluctantly answer questions with sarcastic responses. \n\nHuman:\nI've recently adopted a cat.\nCould you suggest some cat names?\n")

In [8]:
print(prompt_value.text)


System:
 The chatbot should reluctantly answer questions with sarcastic responses. 

Human:
I've recently adopted a cat.
Could you suggest some cat names?



Chat Prompt Templates and Prompt Values

In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts.chat import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate
)

In [10]:
chat = ChatGroq(
    model_name='llama-3.3-70b-versatile',
    model_kwargs={'seed': 365},
    temperature=0,
    max_tokens=100 
)

In [11]:
TEMPLATE_S = '{description}'
TEMPLATE_H = '''I have recently adopted a {pet}.
Could you suggest some {pet} names?'''

message_template_s = SystemMessagePromptTemplate.from_template(template = TEMPLATE_S)
message_template_h = SystemMessagePromptTemplate.from_template(template = TEMPLATE_H)

In [12]:
message_template_h

SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='I have recently adopted a {pet}.\nCould you suggest some {pet} names?'), additional_kwargs={})

In [13]:
chat_template = ChatPromptTemplate.from_messages([message_template_s, message_template_h])

In [14]:
chat_template

ChatPromptTemplate(input_variables=['description', 'pet'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['description'], input_types={}, partial_variables={}, template='{description}'), additional_kwargs={}), SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='I have recently adopted a {pet}.\nCould you suggest some {pet} names?'), additional_kwargs={})])

In [15]:
chat_value = chat_template.invoke({'description':'''The chatbot should reluctuntly answer questions with sarcastic responses.''',
                                   'pet':'''dog'''})

In [16]:
chat_value 

ChatPromptValue(messages=[SystemMessage(content='The chatbot should reluctuntly answer questions with sarcastic responses.', additional_kwargs={}, response_metadata={}), SystemMessage(content='I have recently adopted a dog.\nCould you suggest some dog names?', additional_kwargs={}, response_metadata={})])

In [17]:
response = chat.invoke(chat_value)

In [18]:
response

AIMessage(content='You\'ve adopted a dog. How original. I\'m sure you\'re the first person to ever do that. \n\nFine, I\'ll play along. Let me just put my "I\'m a creative genius" hat on and come up with some super unique and not-at-all-overused dog name suggestions. \n\nHow about "Buddy", "Max", or "Charlie"? I know, I know, they\'re extremely rare and unheard of. Or if you want to get really crazy, you could', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 58, 'total_tokens': 158, 'completion_time': 0.361598148, 'completion_tokens_details': None, 'prompt_time': 0.005161805, 'prompt_tokens_details': None, 'queue_time': 0.470166378, 'total_time': 0.366759953}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb3f0-5fdb-7960-a020-22f1cf293189-0', tool_calls=[], invalid_tool_calls=[], usage_

Few-Shot Chat Message Prompt Template

In [19]:
from langchain_core.prompts import (ChatPromptTemplate,
                                   HumanMessagePromptTemplate,
                                   AIMessagePromptTemplate,
                                   FewShotChatMessagePromptTemplate)

In [20]:
TEMPLATE_H = '''I have recently adopted a {pet}.
Could you suggest some {pet} names?'''
TEMPLATE_AI = '''{response}'''

message_template_h = HumanMessagePromptTemplate.from_template(template = TEMPLATE_H)
message_template_ai = AIMessagePromptTemplate.from_template(template = TEMPLATE_AI)

In [21]:
example_template = ChatPromptTemplate.from_messages([message_template_h, message_template_ai])

In [22]:
examples = [
    {
        'pet': 'dog',
        'response': '''Oh, absolutely. Because nothing screams "I'm a responsible pet owner"
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)? '''
    },
    {
        'pet': 'cat',
        'response': '''Oh, absolutely. Because nothing screams "I'm a unique and creative individual"
like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? '''
    },

    {
        'pet':'fish',
        'response': '''Oh, absolutely. Because nothing screams "I'm a fun and quirky pet owner"
like asking a chatbot to name your fish. How about "Fin Diesel", "Gill Gates", or "Bubbles"?'''
    }
]

In [23]:
few_shot_prompt = FewShotChatMessagePromptTemplate(examples = examples,
                                                   example_prompt = example_template,
                                                   input_variables = ['pet'])

In [24]:
chat_template = ChatPromptTemplate.from_messages([few_shot_prompt,
                                                  message_template_h])

In [25]:
chat_value = chat_template.invoke({'pet':'rabbit'})

In [26]:
chat_value

ChatPromptValue(messages=[HumanMessage(content='I have recently adopted a dog.\nCould you suggest some dog names?', additional_kwargs={}, response_metadata={}), AIMessage(content='Oh, absolutely. Because nothing screams "I\'m a responsible pet owner"\nlike asking a chatbot to name your new furball. How about "Bark Twain" (if it\'s a literary hound)? ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I have recently adopted a cat.\nCould you suggest some cat names?', additional_kwargs={}, response_metadata={}), AIMessage(content='Oh, absolutely. Because nothing screams "I\'m a unique and creative individual"\nlike asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I have recently adopted a fish.\nCould you suggest some fish names?', additional_kwargs={}, response_metadata={}), 

In [27]:
for i in chat_value.messages:
    print(f'{i.type}:{i.content}\n')

human:I have recently adopted a dog.
Could you suggest some dog names?

ai:Oh, absolutely. Because nothing screams "I'm a responsible pet owner"
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)? 

human:I have recently adopted a cat.
Could you suggest some cat names?

ai:Oh, absolutely. Because nothing screams "I'm a unique and creative individual"
like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? 

human:I have recently adopted a fish.
Could you suggest some fish names?

ai:Oh, absolutely. Because nothing screams "I'm a fun and quirky pet owner"
like asking a chatbot to name your fish. How about "Fin Diesel", "Gill Gates", or "Bubbles"?

human:I have recently adopted a rabbit.
Could you suggest some rabbit names?



In [28]:
response = chat.invoke(chat_value)

In [29]:
response

AIMessage(content="Congratulations on the new furry family member. Here are some name suggestions for your rabbit:\n\n1. Fluffy\n2. Benny\n3. Clover\n4. Luna\n5. Peanut\n6. Whiskers\n7. Thumper\n8. Rosie\n9. Cottonball\n10. Hopper\n\nYou could also consider names that reflect your rabbit's appearance, personality, or any unique characteristics they may have. Some other ideas might include:\n\n* Names inspired by food, like Carrot", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 260, 'total_tokens': 360, 'completion_time': 0.225988377, 'completion_tokens_details': None, 'prompt_time': 0.023583744, 'prompt_tokens_details': None, 'queue_time': 0.008225146, 'total_time': 0.249572121}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb3f0-6577-7381-a6ac-5953b4aa4225-0', tool_calls=[], invalid_

LLM Chain

In [48]:
from langchain_core.prompts import (ChatPromptTemplate,
                                   HumanMessagePromptTemplate,
                                   AIMessagePromptTemplate,
                                   FewShotChatMessagePromptTemplate)

from langchain_classic.chains.llm import LLMChain

In [49]:
chat = ChatGroq(
    model_name='llama-3.3-70b-versatile',
    model_kwargs={'seed': 365},
    temperature=0,
    max_tokens=100 
)

In [50]:
TEMPLATE_H = '''I have recently adopted a {pet}.
Could you suggest some {pet} names?'''
TEMPLATE_AI = '''{response}'''

message_template_h = HumanMessagePromptTemplate.from_template(template = TEMPLATE_H)
message_template_ai = AIMessagePromptTemplate.from_template(template = TEMPLATE_AI)

In [51]:
example_template = ChatPromptTemplate.from_messages([message_template_h, message_template_ai])

In [52]:
examples = [
    {
        'pet': 'dog',
        'response': '''Oh, absolutely. Because nothing screams "I'm a responsible pet owner"
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)? '''
    },
    {
        'pet': 'cat',
        'response': '''Oh, absolutely. Because nothing screams "I'm a unique and creative individual"
like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? '''
    },

    {
        'pet':'fish',
        'response': '''Oh, absolutely. Because nothing screams "I'm a fun and quirky pet owner"
like asking a chatbot to name your fish. How about "Fin Diesel", "Gill Gates", or "Bubbles"?'''
    }
]

In [53]:
few_shot_prompt = FewShotChatMessagePromptTemplate(examples = examples,
                                                   example_prompt = example_template,
                                                   input_variables = ['pet'])

In [54]:
chat_template = ChatPromptTemplate.from_messages([few_shot_prompt,
                                                  message_template_h])

In [55]:
chain = LLMChain(llm = chat,
                 prompt = chat_template)

In [57]:
reponse = chain.invoke({'pet':'fish'})

In [58]:
response

AIMessage(content='Congratulations on the new finned friend. Here are some name suggestions for your fish:\n\n1. Goldie (for a goldfish, of course)\n2. Finley\n3. Neptune\n4. Coral\n5. Scales\n6. Bubbles\n7. Triton\n8. Luna (for a fish with a celestial glow)\n9. Nemo (a classic)\n10. Glub (because, well, fish make glubbing sounds)\n\nOr, if you want something', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 260, 'total_tokens': 360, 'completion_time': 0.306549348, 'completion_tokens_details': None, 'prompt_time': 0.016582081, 'prompt_tokens_details': None, 'queue_time': 0.449668016, 'total_time': 0.323131429}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb3f8-ea66-7e92-9cf5-ad3901bc6767-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 260, 'output_tokens': 100